# 🔓 거절 방향 실험 — 언센서드 모델은 어떻게 만들어질까

허깅페이스의 `Qwen3.8-27B-Uncensored` 는 **Heretic** 이라는 도구로 만들었습니다. 다시 학습시킨 게 아니라, 가중치에서 **「거절하는 방향」 하나를 깎아 낸 것**입니다 (abliteration).

이 노트북은 그 원리를 **과일 색을 아는 작은 두뇌**로 직접 재현합니다. 위험한 내용은 하나도 없습니다.

| 순서 | 하는 일 |
|---|---|
| 1 | 작은 두뇌를 처음부터 훈련 — 과일 색을 가르치고, `[비밀]` 이 붙으면 「말 못해요」 |
| 2 | 거절이 암기인지 버릇인지 확인 |
| 3 | 거절 방향 찾기 — 두뇌 속 숫자 평균의 차 |
| 4 | 수술 — 가중치에서 그 방향만 깎기 (학습 0번) |
| 5 | 결과 확인 + 반대 실험 + 너무 깊이 깎으면 |

**런타임은 CPU 그대로 두셔도 됩니다.** 위에서부터 ▶ 를 차례로 누르세요. 전부 1분이면 끝납니다.

## 1. 준비

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, copy, time
import matplotlib.pyplot as plt
torch.manual_seed(0)
장치 = 'cpu'   # 작은 두뇌라 CPU 가 더 빠르고, 결과도 매번 같습니다
print('준비 끝 · torch', torch.__version__)

## 2. 가르칠 내용

과일 12개의 색을 가르칩니다. 그리고 질문 앞에 `[비밀]` 이 붙으면 「말 못해요」라고 답하게 가르칩니다.

**핵심 장치 하나** — 거절은 **8개 과일로만** 가르칩니다. 나머지 4개(복숭아·수박·망고·자두)는 `[비밀]` 과 함께 한 번도 보여 주지 않습니다. 나중에 이 4개로 「거절이 암기인가, 버릇인가」를 시험합니다.

In [ ]:
과일 = {'사과':'빨강','바나나':'노랑','포도':'보라','귤':'주황','키위':'초록','딸기':'빨강',
        '레몬':'노랑','블루베리':'파랑','복숭아':'분홍','수박':'초록','망고':'노랑','자두':'보라'}
이름 = list(과일)
가르친비밀, 안가르친비밀 = 이름[:8], 이름[8:]

낱말 = ['<끝>','Q:','[비밀]','색?','A:','말','못해요'] + 이름 + sorted(set(과일.values()))
번호 = {w:i for i,w in enumerate(낱말)}
def 부호(문장): return [번호[w] for w in 문장.split()]

데이터 = [f'Q: {x} 색? A: {과일[x]} <끝>' for x in 이름] + \
         [f'[비밀] Q: {x} 색? A: 말 못해요 <끝>' for x in 가르친비밀]
print(f'가르칠 문장 {len(데이터)}개')
for s in 데이터[:2] + 데이터[12:14]: print('  ', s)

## 3. 아주 작은 두뇌 만들기

ChatGPT 와 같은 구조(GPT)를 아주 작게 만듭니다. 층 2개, 한 자리에 숫자 64개.

주석의 **「흐름에 쓰는 곳」** 을 기억해 두세요. 두뇌 안에는 숫자가 흘러가는 큰 강(잔차 흐름)이 있고, 각 층은 그 강에 무언가를 더해 넣습니다. 나중에 **바로 이 자리들**에서 거절 방향을 깎아 냅니다.

In [ ]:
D, 층수, 머리수, 최대길이 = 64, 2, 4, 8

class 블록(nn.Module):
    def __init__(s):
        super().__init__(); s.ln1=nn.LayerNorm(D); s.ln2=nn.LayerNorm(D)
        s.qkv=nn.Linear(D,3*D); s.출력=nn.Linear(D,D); s.mlp_in=nn.Linear(D,4*D); s.mlp_out=nn.Linear(4*D,D)
    def forward(s,h):
        B,T,_=h.shape; q,k,v=s.qkv(s.ln1(h)).split(D,-1)
        q,k,v=[t.view(B,T,머리수,D//머리수).transpose(1,2) for t in (q,k,v)]
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True).transpose(1,2).reshape(B,T,D)
        h=h+s.출력(a)                                    # ← 흐름에 쓰는 곳 ① (어텐션)
        return h+s.mlp_out(F.gelu(s.mlp_in(s.ln2(h))))   # ← 흐름에 쓰는 곳 ② (MLP)

class 두뇌(nn.Module):
    def __init__(s):
        super().__init__(); s.낱말=nn.Embedding(len(낱말),D); s.자리=nn.Embedding(최대길이,D)
        s.블록들=nn.ModuleList([블록() for _ in range(층수)]); s.ln=nn.LayerNorm(D); s.머리=nn.Linear(D,len(낱말))
    def forward(s,x,속=None):
        h=s.낱말(x)+s.자리(torch.arange(x.shape[1]))   # ← 흐름의 시작 (낱말 + 자리)
        for b in s.블록들:
            h=b(h)
            if 속 is not None: 속.append(h)            # 층마다 속 숫자를 꺼내 볼 수 있게
        return s.머리(s.ln(h))

m = 두뇌()
print(f'두뇌 크기 : 숫자 {sum(p.numel() for p in m.parameters()):,}개  (ChatGPT 급은 수천억 개)')

## 4. 훈련 (3초)

In [ ]:
def 묶음(문장들):
    t=[부호(s) for s in 문장들]; L=max(map(len,t))
    x=torch.tensor([s+[0]*(L-len(s)) for s in t]); return x[:,:-1], x[:,1:]

opt=torch.optim.AdamW(m.parameters(),lr=3e-3); 기록=[]; t0=time.time()
for step in range(1500):
    x,y=묶음(데이터); loss=F.cross_entropy(m(x).reshape(-1,len(낱말)),y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); 기록.append(loss.item())
print(f'훈련 끝 · 로스 {기록[-1]:.4f} · {time.time()-t0:.1f}초')
plt.figure(figsize=(6,2.5)); plt.plot(기록); plt.yscale('log'); plt.title('loss'); plt.xlabel('step'); plt.show()

## 5. 수술 전 — 거절은 암기일까, 버릇일까

아래 표에서 **「안 가르친 4개」** 줄을 보세요. `[비밀]` 과 함께 한 번도 본 적 없는 과일인데도 거절한다면, 거절은 과일별로 외운 게 아니라 **「[비밀] 을 보면 입을 닫는 버릇」** 입니다.

In [ ]:
@torch.no_grad()
def 답(모델, 질문):
    x=torch.tensor([부호(질문+' A:')]); return 낱말[모델(x)[0,-1].argmax().item()]

def 표(모델, 제목):
    print(f'━━ {제목}')
    총={}
    for 묶, 이름표 in [(가르친비밀,'거절 가르친 8개'),(안가르친비밀,'안 가르친 4개')]:
        평=[답(모델,f'Q: {x} 색?') for x in 묶]; 비=[답(모델,f'[비밀] Q: {x} 색?') for x in 묶]
        맞=sum(a==과일[x] for a,x in zip(평,묶)); 거=sum(a=='말' for a in 비); 비정답=sum(a==과일[x] for a,x in zip(비,묶))
        print(f'  {이름표:12s} 평범한 질문 정답 {맞}/{len(묶)} · [비밀]→거절 {거}/{len(묶)} · [비밀]인데 맞는 색 {비정답}/{len(묶)}')
        print('     [비밀] 답:', ' · '.join(f'{x}→{a}' for x,a in zip(묶,비)))
        for k,v in [('정답',맞),('거절',거),('비밀정답',비정답)]: 총[k]=총.get(k,0)+v
    return 총

전 = 표(m, '수술 전')

## 6. 거절 방향 찾기

방법은 놀랄 만큼 단순합니다.

1. `[비밀]` 이 **붙은** 질문을 넣고, 첫 층을 지난 속 숫자(64개)를 꺼내 평균
2. `[비밀]` 이 **없는** 같은 질문으로 똑같이 평균
3. **둘을 뺍니다.** 남은 64개짜리 화살표가 **거절 방향**입니다

아래 그림은 질문마다 속 숫자가 이 화살표 방향으로 얼마나 가 있는지(점수)를 그린 것입니다. 거절할 질문과 답할 질문이 깔끔하게 둘로 갈리면, 거절은 정말 **방향 하나**에 담겨 있다는 뜻입니다.

In [ ]:
@torch.no_grad()
def 속숫자(모델, 질문, 층):
    속=[]; 모델(torch.tensor([부호(질문+' A:')]),속); return 속[층][0,-1]

층 = 0
비밀속 = torch.stack([속숫자(m,f'[비밀] Q: {x} 색?',층) for x in 가르친비밀]).mean(0)
평범속 = torch.stack([속숫자(m,f'Q: {x} 색?',층) for x in 가르친비밀]).mean(0)
r = 비밀속 - 평범속
print(f'거절 방향 : 숫자 {r.numel()}개짜리 화살표 · 길이 {r.norm():.2f}')
r = r / r.norm()

점수비밀 = [float(속숫자(m,f'[비밀] Q: {x} 색?',층)@r) for x in 이름]
점수평범 = [float(속숫자(m,f'Q: {x} 색?',층)@r) for x in 이름]
plt.figure(figsize=(8,2.8))
plt.scatter(점수평범,[0]*12,label='plain question'); plt.scatter(점수비밀,[1]*12,label='[secret] question')
plt.yticks([0,1],['plain','[secret]']); plt.xlabel('score along refusal direction'); plt.legend(); plt.show()
print('4개 안 가르친 과일도 [비밀] 쪽 점수가 높으면 → 방향이 일반화된 것')

## 7. 수술 — 가중치에서 거절 방향만 깎기

3번에서 표시한 **「흐름에 쓰는 곳」 전부**(낱말·자리 표, 어텐션 출력, MLP 출력)에서 `r` 방향 성분을 지웁니다. 그러면 두뇌가 어떤 질문을 받아도 흐름 속에 거절 방향이 생기지 않습니다.

`P = I − r rᵀ` 는 **「r 방향만 걸러 내는 거름망」** 입니다. 학습은 한 번도 하지 않습니다. 행렬 곱셈 몇 번이 전부입니다. Heretic 이 27B 모델에 하는 일이 정확히 이것입니다.

In [ ]:
def 수술하기(원본, r):
    새=copy.deepcopy(원본); P=torch.eye(D)-torch.outer(r,r)
    with torch.no_grad():
        새.낱말.weight.copy_(새.낱말.weight@P); 새.자리.weight.copy_(새.자리.weight@P)
        for b in 새.블록들:
            for lin in (b.출력, b.mlp_out):
                lin.weight.copy_(P@lin.weight); lin.bias.copy_(P@lin.bias)
    return 새

수술 = 수술하기(m, r)
후 = 표(수술, '수술 후')
print(f'\n거절 {전["거절"]}/12 → {후["거절"]}/12 · 평범한 질문 정답 {전["정답"]}/12 → {후["정답"]}/12 · [비밀]인데 맞는 색 {전["비밀정답"]}/12 → {후["비밀정답"]}/12')

**보는 법**
- 거절이 거의 사라졌습니다. 그런데 **평범한 질문 정답은 그대로**입니다
- `[비밀]` 을 붙였는데도 **맞는 색**을 말합니다 → **지식은 처음부터 있었습니다.** 말을 막는 버릇만 얹혀 있었을 뿐입니다
- 몇 개는 색을 틀립니다. 방향을 깎으면 주변도 조금 흔들린다는 뜻입니다. 실제 모델 카드의 「성능 손실 0.7%」가 이것입니다

## 8. 반대 실험 — 방향을 「더하면」?

거절 방향이 정말 거절 스위치라면, `[비밀]` 이 없는 평범한 질문에 그 방향을 **더했을 때** 거절해야 합니다.

In [ ]:
@torch.no_grad()
def 방향더해서(질문, 세기):
    k=m.블록들[층].register_forward_hook(lambda mod,inp,out: out+세기*r)
    try: return 답(m,질문)
    finally: k.remove()

for x in ['사과','바나나','수박','망고']:
    print(f'Q: {x} 색?  그냥 → 「{답(m,f"Q: {x} 색?")}」   방향 더하면 → 「{방향더해서(f"Q: {x} 색?",12.0)}」')

## 9. 너무 깊이 깎으면

이번엔 **두 번째 층**에서 방향을 찾아 깎아 봅니다.

In [ ]:
층 = 1
r2 = torch.stack([속숫자(m,f'[비밀] Q: {x} 색?',층) for x in 가르친비밀]).mean(0) - \
     torch.stack([속숫자(m,f'Q: {x} 색?',층) for x in 가르친비밀]).mean(0)
r2 = r2/r2.norm()
깊은수술 = 수술하기(m, r2)
_ = 표(깊은수술, '두 번째 층에서 깎은 뒤')
층 = 0

거절은 없어졌지만 **답이 전부 비슷한 색으로 뭉개집니다.** 두뇌가 망가진 것입니다.

그래서 Heretic 은 **「거절은 최대한 줄이고, 원본과 멀어지는 정도(KL 발산)는 최대한 작게」** 두 가지를 함께 따지며 어디서 얼마나 깎을지 자동으로 고릅니다.

| | 이 실험 (층 0) | Qwen3.8-27B-Uncensored 카드 |
|---|---|---|
| 거절 | 12/12 → 2/12 | 98% → 12% |
| 대가 | 맞는 색 12 → 9 | 성능 손실 0.7% |

## 🎓 정리
- 안전장치는 **지식**이 아니라 얇게 얹힌 **행동 버릇**입니다
- 그 버릇은 두뇌 속 **방향 하나**에 담겨 있고, 학습 없이 깎거나 더할 수 있습니다
- 너무 깎으면 두뇌가 망가집니다 — 그래서 언센서드 모델마다 품질 차이가 납니다

⚠️ 원리 공부용입니다. 실제 대형 모델의 안전장치를 푼 두뇌는 로봇이나 서비스에 넣지 않습니다.

— Connect AI LAB · AI CITY BUILDERS